# Problem 3. 스마트폰 센서로 행동 인식하기

## 문제 개요
헬스케어 앱은 허리에 찬 스마트폰의 센서만으로 사용자가 **지금 무엇을 하고 있는지** 알아내려 합니다.
2.56초 길이의 가속도·자이로 신호 구간을 보고 6가지 행동 중 하나로 분류하세요.

## 파일 구조
```
Problem3/
├── problem3.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train_signals.npy    학습 신호 (7352, 128, 6)
    ├── test_signals.npy     테스트 신호 (2947, 128, 6)
    ├── train.csv            id, subject, activity
    ├── test.csv             id, subject
    └── sample_submission.csv
```

## 데이터
| 파일 | 설명 |
|---|---|
| `train_signals.npy` | 신호 배열 (구간 수, 128 시점, 6 채널). `train.csv`와 행 순서가 같음 |
| `test_signals.npy` | 테스트 신호 배열. `test.csv`와 행 순서가 같음 |
| `train.csv` | id, subject(참가자 번호), **activity(예측 대상)** |
| `test.csv` | id, subject |

- 6개 채널: `[가속도 x, y, z, 자이로 x, y, z]` (가속도는 중력 포함, 단위 g / 자이로 단위 rad/s)
- 50Hz로 측정, 한 구간 = 128 시점 = 2.56초, **이웃 구간끼리 50% 겹침**
- 행동: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING

### 데이터 특징
- 학습(21명)과 테스트(9명)는 **서로 다른 사람**입니다.
- 각 파일은 **사람별로 시간 순서대로** 정렬되어 있습니다.
- 출처: Human Activity Recognition Using Smartphones (UCI Machine Learning Repository, CC BY 4.0)의 원시 신호

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem3/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 높을수록 좋습니다.

## 초기 코드 (베이스라인)
아래 베이스라인은 채널별 평균·표준편차 12개로 로지스틱 회귀를 합니다. 동작은 하지만 점수가 낮습니다.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv")
Xtr = np.load(f"{DATA_DIR}/train_signals.npy"); Xte = np.load(f"{DATA_DIR}/test_signals.npy")   # (구간 수, 128, 6)
def feat(X): return np.hstack([X.mean(axis=1), X.std(axis=1)])   # 채널별 평균·표준편차 12개
clf = LogisticRegression(max_iter=1000).fit(feat(Xtr), train.activity)
pd.DataFrame({"id": test.id, "activity": clf.predict(feat(Xte))}).to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| id | activity |
|---|---|
| 10000 | STANDING |
| 10001 | STANDING |

- 컬럼명 `id`, `activity`, activity는 위 6개 이름 그대로
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()